# CarbonSifr — EXIOBASE Emission-Factor Activity Mapper

Fine-tuning **Qwen2.5-14B-Instruct** with **LoRA** to map procurement items to one of 83 EXIOBASE emission-factor activities.

**Author:** Ibrahim Ahmad · June 2026

---

### Pipeline
1. **Data & taxonomy** — load the 20 gold labels, 2 000-item pool, 83-activity catalog
2. **Prompt assembly** — rule book (`[GOLD]` + `[EXTENDED]`) + few-shot + activity catalog → system prompt
3. **Silver labeling** — Qwen2.5-**32B** teacher labels 600 items; **Claude** audits 100; **Gemini** cross-checks 500
4. **Dataset assembly** — resolve teacher disagreements → `train / val / test_silver` splits
5. **Baselines** — embedding-NN · zero-shot · base+rule-book
6. **Fine-tune** — LoRA (Unsloth + TRL) on the silver train set
7. **Evaluation** — fine-tuned vs all baselines on gold-20 + silver-52
8. **Demo** — single-item classifier

### Reproduction
The notebook **loads cached artifacts from Google Drive by default** so it runs end-to-end in ~10 min.
Set the flags in §0 to regenerate the silver labels (`REGENERATE_SILVER`) or retrain the adapter (`RETRAIN`).

> Requires an **A100** runtime (Runtime → Change runtime type → A100).


## 0 · Environment setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, json, re, time, gc
import pandas as pd
import numpy as np

ROOT       = '/content/drive/MyDrive/carbonsifr'
DRIVE_ROOT = ROOT  # alias used in a few places
for d in ['data', 'data/splits', 'prompts', 'artifacts', 'silver_labels', 'checkpoints']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

# --- Reproduction flags -------------------------------------------------
# Both default to False → load the cached outputs produced by the full run.
REGENERATE_SILVER = False   # True  → re-run Qwen-32B labeling + Claude/Gemini cross-check
RETRAIN           = True   # True  → re-train the LoRA adapter (~40 min on A100)
# ------------------------------------------------------------------------
print('ROOT =', ROOT)

ROOT = /content/drive/MyDrive/carbonsifr


**Dependencies.** Versions are pinned for Unsloth compatibility — if Colab prompts to restart after install, do **Runtime → Restart session** (not *Disconnect*, to keep the A100) and re-run from here.

In [ ]:
import os
_SENTINEL = "/content/_carbonsifr_deps_ready"
if not os.path.exists(_SENTINEL):
    !pip install -q unsloth
    !pip install -q "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0"
    open(_SENTINEL, "w").close()
    print("Dependencies installed — restarting runtime to load new binaries.")
    print("When it reconnects: Runtime -> Run all (this cell will skip).")
    os.kill(os.getpid(), 9)   # force a one-time kernel restart
else:
    print("Dependencies ready (post-restart). Continuing.")

In [4]:
import torch
from unsloth import FastLanguageModel
print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch 2.10.0+cu128 | CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## 1 · Data & taxonomy

`Data.xlsx` has three sheets: the **83-activity catalog**, the **2 000-item** procurement pool, and the **20 gold** assignments. The gold labels are the held-out headline test set — **never used for training**.

In [5]:
DATA_PATH  = f'{ROOT}/data/Data.xlsx'
activities = pd.read_excel(DATA_PATH, sheet_name='Exiobase Activities')
items      = pd.read_excel(DATA_PATH, sheet_name='Items Sample')
gold       = pd.read_excel(DATA_PATH, sheet_name='Exiobase Assignment Sample')
print(f'Activities: {len(activities)} classes | Items pool: {len(items)} | Gold labels: {len(gold)}')

Activities: 83 classes | Items pool: 2000 | Gold labels: 20


In [6]:
# Headline test set: gold labels joined to their descriptions
gold_test = gold.merge(
    items[['item_id','item_description','main_category','category_3']],
    on='item_id', how='left')
gold_test.to_csv(f'{ROOT}/data/splits/test_gold.csv', index=False)
print('gold_test:', gold_test.shape)
gold_test[['item_description','activity','confidence_level']].head()

gold_test: (20, 7)


,item_description,activity,confidence_level
0,NESCAFE COFFEE EXTRA FORTE 12x230 GMS - 1X1CARTON,Food products nec,0.70
1,Supply of Window Blinds,Furniture; other manufactured goods n.e.c. (36),0.80
2,ESG 7 GenAI Images SOW,Computer and related services (72),0.80
3,EXPANDING FILE 13 POCKET A4SIZE,Paper and paper products,0.95
4,fruit-fresh strawberry,"Vegetables, fruit, nuts",0.95


In [7]:
# Activity structures used by the prompts + metrics
activity_list = "\n".join(f"- {a}" for a in activities['activity_name'].tolist())   # flat (zero-shot)

ACTIVITY_LIST = ""                                                                    # sector-grouped (rule-book)
for sector, group in activities.sort_values(['activity_sector','activity_name']).groupby('activity_sector'):
    ACTIVITY_LIST += f"\n## {sector}\n"
    for a in group['activity_name']:
        ACTIVITY_LIST += f"- {a}\n"

sector_map = dict(zip(activities['activity_name'], activities['activity_sector']))     # activity → 16-class sector

n_cov = gold['activity'].nunique()
print(f'Gold examples cover {n_cov}/83 activities → {83 - n_cov} classes extended from EXIOBASE definitions in the rule book')

Gold examples cover 20/83 activities → 63 classes extended from EXIOBASE definitions in the rule book


## 2 · Prompt assembly

The system prompt = **rule book** (≈50 rules, each tagged `[GOLD]` or `[EXTENDED]`) + **3 synthetic few-shot examples** + the **full 83-activity catalog**. The model must return strict JSON: `{activity, reason, confidence_level}`.

In [8]:
with open(f'{ROOT}/prompts/rule_book.md') as f: RULE_BOOK      = f.read()
with open(f'{ROOT}/prompts/few_shot.md')  as f: FEW_SHOT_BLOCK = f.read()
print(f'Rule book ≈ {len(RULE_BOOK)//4} tokens | Few-shot ≈ {len(FEW_SHOT_BLOCK)//4} tokens')

Rule book ≈ 3060 tokens | Few-shot ≈ 429 tokens


In [9]:
# Build the full system prompt by stitching together all the pieces

SYSTEM_PROMPT = f"""You are an emission-factor classification assistant. Your task is to map each procurement item to exactly one EXIOBASE activity.

For every item, respond with ONLY a JSON object in this exact schema (no markdown, no surrounding text):
{{"activity": "<exact name from the valid activities list>", "reason": "<1-2 sentence justification citing the relevant rule>", "confidence_level": <float between 0 and 1>}}

The "activity" value MUST be one of the 83 valid activity names listed at the end of this prompt — character-for-character.

Below is the rule book you must follow, the worked examples that demonstrate the expected output style, and the full list of valid activities.

---

# RULE BOOK

{RULE_BOOK}

---

# WORKED EXAMPLES

{FEW_SHOT_BLOCK}

---

# VALID ACTIVITIES (output must match one of these exactly)

{ACTIVITY_LIST}
"""

USER_TEMPLATE = """Item description: {item_description}
Buyer category: {main_category}
Sub-category: {category_3}"""


def build_messages(item_description, main_category=None, category_3=None):
    """Assemble the chat messages list for any LLM call."""
    user_content = USER_TEMPLATE.format(
        item_description=item_description,
        main_category=main_category if main_category else "unspecified",
        category_3=category_3 if category_3 else "unspecified",
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


# Sanity check: assemble a prompt for one real item and inspect
sample_item = items.iloc[0]
msgs = build_messages(
    sample_item['item_description'],
    sample_item['main_category'],
    sample_item['category_3'],
)

print(f"System prompt length: {len(msgs[0]['content'])} chars (~{len(msgs[0]['content'])//4} tokens)")
print(f"User message length:  {len(msgs[1]['content'])} chars (~{len(msgs[1]['content'])//4} tokens)")
print()
print("--- USER MESSAGE PREVIEW ---")
print(msgs[1]['content'])
print()
print("--- SYSTEM PROMPT (last 300 chars, to confirm activity list at end) ---")
print(msgs[0]['content'][-300:])

System prompt length: 18446 chars (~4611 tokens)
User message length:  109 chars (~27 tokens)

--- USER MESSAGE PREVIEW ---
Item description: TORMAX EAGLE RADAR
Buyer category: Facility Management
Sub-category: DOORS/WINDOWS/SHUTTERS

--- SYSTEM PROMPT (last 300 chars, to confirm activity list at end) ---
sport equipment
- Motor vehicles, trailers and semi-trailers (34)
- Other transport equipment (35)

## Wood, paper & publishing
- Paper and paper products
- Printed matter and recorded media (22)
- Wood and products of wood and cork (except furniture); articles of straw and plaiting materials (20)




In [10]:
import json, re

def parse_json_output(text):
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    m = re.search(r'\{[^{}]*"activity"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    text2 = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.MULTILINE)
    try:
        return json.loads(text2)
    except json.JSONDecodeError:
        pass
    return None

print("✓ parse_json_output defined")

✓ parse_json_output defined


A **rule-free** prompt is also defined — it measures the raw model with only the activity list, as an un-leaked baseline (the rule book is partly derived from the gold items, so a rule-book score on gold-20 is optimistic).

In [11]:
SYSTEM_PROMPT_ZEROSHOT = f"""You are a carbon accounting expert. Map the procurement item to exactly one EXIOBASE activity.

Valid activities:
{activity_list}

Respond with valid JSON only:
{{"activity": "<activity name>", "reason": "<one sentence>", "confidence_level": <0.0-1.0>}}"""

def build_messages_zeroshot(item_description, main_category=None, category_3=None):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_ZEROSHOT},
        {"role": "user", "content": USER_TEMPLATE.format(
            item_description=item_description,
            main_category=main_category or "unspecified",
            category_3=category_3 or "unspecified"
        )}
    ]

In [12]:
# Persist the assembled prompt for the repo
with open(f'{ROOT}/prompts/system_prompt.txt','w') as f: f.write(SYSTEM_PROMPT)
with open(f'{ROOT}/prompts/user_template.txt','w') as f: f.write(USER_TEMPLATE)
print('Saved system_prompt.txt + user_template.txt')

Saved system_prompt.txt + user_template.txt


## 3 · Silver-label generation  *(cached — set `REGENERATE_SILVER=True` to rebuild)*

Training labels can't come from the 20 gold items (too few, and reserved for test). Instead an **open-source teacher** labels a stratified 600-item sample, and two independent reviewers check it:

| Stage | Model | Role |
|---|---|---|
| Primary labeler | **Qwen2.5-32B-Instruct** (4-bit) | labels all 600 items under the rule book |
| Quality audit | **Claude** | independently re-labels a random **100** → agreement signal |
| Cross-check | **Gemini 2.5 Pro** (AI Studio) | labels the other **500** → disagreement resolution |

Claude's audit labels never enter training (auditor, not labeler). Gemini's labels are used only to **flag** Qwen disagreements for review.

In [13]:
# 600-item stratified pool (load the exact IDs used in the run if present)
ids_path = f'{ROOT}/silver_labels/silver_pool_ids.csv'
if os.path.exists(ids_path):
    silver_pool = items.merge(pd.read_csv(ids_path), on='item_id').reset_index(drop=True)
    print('Loaded silver_pool IDs:', len(silver_pool))
else:
    GOLD_IDS = set(gold['item_id'])
    pool = items[~items['item_id'].isin(GOLD_IDS)].copy()
    pool['main_category'] = pool['main_category'].fillna('UNKNOWN')
    silver_pool = (pool.groupby('main_category', group_keys=False)
        .apply(lambda g: g.sample(n=max(1, min(len(g), round(600*len(g)/len(pool)))), random_state=42))
        .reset_index(drop=True))
    silver_pool = silver_pool.sample(n=min(600, len(silver_pool)), random_state=42).reset_index(drop=True)
    silver_pool[['item_id']].to_csv(ids_path, index=False)
    print('Sampled silver_pool:', len(silver_pool))

Loaded silver_pool IDs: 600


In [14]:
# --- Qwen-32B teacher labeling (load cache, else generate) ---
qwen_path = f'{ROOT}/silver_labels/qwen_labels.csv'
if os.path.exists(qwen_path) and not REGENERATE_SILVER:
    qwen_df = pd.read_csv(qwen_path)
    print(f'Loaded cached Qwen labels: {len(qwen_df)} rows | parse rate {qwen_df["parse_ok"].mean():.0%}')
else:
    from tqdm.notebook import tqdm
    teacher_qwen, tok_qwen = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-32B-Instruct-bnb-4bit", max_seq_length=8192, load_in_4bit=True)
    FastLanguageModel.for_inference(teacher_qwen)
    tok_qwen.padding_side = "left"
    if tok_qwen.pad_token is None: tok_qwen.pad_token = tok_qwen.eos_token
    STOP = [tok_qwen.eos_token_id, tok_qwen.convert_tokens_to_ids("<|im_end|>")]

    BATCH, rows = 4, []
    for s in tqdm(range(0, len(silver_pool), BATCH), desc="Qwen-32B"):
        b = silver_pool.iloc[s:s+BATCH]
        prompts = [tok_qwen.apply_chat_template(
            build_messages(r.item_description, r.main_category, r.category_3),
            tokenize=False, add_generation_prompt=True) for r in b.itertuples()]
        inp = tok_qwen(prompts, return_tensors="pt", padding=True, truncation=True, max_length=6000).to("cuda")
        L = inp['input_ids'].shape[1]
        with torch.no_grad():
            out = teacher_qwen.generate(**inp, max_new_tokens=150, do_sample=False,
                                        eos_token_id=STOP, pad_token_id=tok_qwen.eos_token_id)
        for i, r in enumerate(b.itertuples()):
            resp = tok_qwen.decode(out[i, L:], skip_special_tokens=True).strip()
            p = parse_json_output(resp)
            rows.append({'item_id': r.item_id, 'item_description': r.item_description,
                'main_category': r.main_category,
                'category_3': r.category_3 if pd.notna(r.category_3) else None,
                'raw_response': resp,
                'qwen_activity':   p.get('activity')          if p else None,
                'qwen_reason':     p.get('reason')            if p else None,
                'qwen_confidence': p.get('confidence_level')  if p else None,
                'parse_ok': p is not None})
    qwen_df = pd.DataFrame(rows)
    qwen_df.to_csv(qwen_path, index=False)
    print(f'Qwen labeling done: {len(qwen_df)} | parse rate {qwen_df["parse_ok"].mean():.0%}')

Loaded cached Qwen labels: 600 rows | parse rate 100%


In [15]:
# --- Claude audit (canonical = cached CSV with the exact 100 audited item_ids) ---
claude_audit_df = pd.read_csv(f'{ROOT}/silver_labels/claude_audit.csv')
print('Claude audit labels:', len(claude_audit_df))

Claude audit labels: 100


<details><summary><b>Provenance — Claude audit raw responses + parser</b> (runs only if <code>REGENERATE_SILVER=True</code>)</summary>

The 100-item audit was done in four 25-item batches. The raw Claude JSON and the parser are preserved below for transparency.</details>

In [16]:
if REGENERATE_SILVER:
    # Exact audited set + order, as saved during the original run
    audit_sample = pd.read_csv(f'{ROOT}/silver_labels/audit_sample_with_qwen.csv')

    batch_1_json = """[1] {"activity": "Food products nec", "reason": "Beyond Meat is a packaged plant-based food product not matching a specific sub-category (not dairy, meat, fish, fresh produce, sugar, rice, or oils). Per the food rule, packaged food without a specific sub-category maps to 'Food products nec'.", "confidence_level": 0.85}
    [2] {"activity": "Furniture; other manufactured goods n.e.c. (36)", "reason": "Letter tray with cork liner is a multi-material composite manufactured good (metal/plastic body with cork accessory). Per the composite-goods override, multi-material manufactured items map to 'Furniture; other manufactured goods n.e.c. (36)'.", "confidence_level": 0.7}
    [3] {"activity": "Paper and paper products", "reason": "Bouquet wrapping sheets are typically single-material paper. Per the material rule, single-material paper items map to 'Paper and paper products'. Some ambiguity remains as decorative wrapping can also be plastic film.", "confidence_level": 0.6}
    [4] {"activity": "Other business services (74)", "reason": "Brochures and stickers in the PROMOTIONAL ITEMS category are marketing collateral, not media for sale. Per the service rule, advertising and printed marketing materials map to 'Other business services (74)'.", "confidence_level": 0.75}
    [5] {"activity": "Other business services (74)", "reason": "Branded exhibition booth construction is a promotional/marketing service. Per the service rule, branding and advertising services map to 'Other business services (74)'.", "confidence_level": 0.7}
    [6] {"activity": "Other services (93)", "reason": "Voucher cards used as employee recognition awards do not fit a specific product or service activity. Per the fallback hierarchy, generic services not matching a specific category map to 'Other services (93)'.", "confidence_level": 0.5}
    [7] {"activity": "Paper and paper products", "reason": "An 80gsm plotter roll is single-material paper stock. Per the material rule, single-material paper items map to 'Paper and paper products'. The gsm specification confirms paper.", "confidence_level": 0.95}
    [8] {"activity": "Other business services (74)", "reason": "Excess printing copy charges describe a print service overage fee, not the physical paper. Per the service rule, business support and printing services map to 'Other business services (74)'.", "confidence_level": 0.65}
    [9] {"activity": "Construction work (45)", "reason": "Dismantling is construction/renovation work. Per the materials-vs-work rule, building work and dismantling services map to 'Construction work (45)'. The storage component is secondary to the dismantling service.", "confidence_level": 0.55}
    [10] {"activity": "Textiles (17)", "reason": "Parasol covers are typically single-material fabric. Per the material rule, textile fabric items (not garments, not flags) map to 'Textiles (17)'.", "confidence_level": 0.7}
    [11] {"activity": "Other services (93)", "reason": "'Fixed Fee' with only GRN reference numbers provides no item or service detail. Per the under-specified rule, ambiguous descriptions get low confidence; fallback to 'Other services (93)'.", "confidence_level": 0.2}
    [12] {"activity": "Public administration and defence services; compulsory social security services (75)", "reason": "A 'Tourism fee' is typically a government-imposed tax or levy on accommodation. Per the service rule, government administration and compulsory fees map to 'Public administration (75)'. Some ambiguity as it could be a hotel charge.", "confidence_level": 0.55}
    [13] {"activity": "Textiles (17)", "reason": "Satin ribbon is a single-material textile product. Per the material rule, textile fabric items (not garments, not flags) map to 'Textiles (17)'.", "confidence_level": 0.9}
    [14] {"activity": "Wearing apparel; furs (18)", "reason": "Uniform trousers are a garment. Per the apparel rule, clothing and uniforms map to 'Wearing apparel; furs (18)'.", "confidence_level": 0.95}
    [15] {"activity": "Wood and products of wood and cork (except furniture); articles of straw and plaiting materials (20)", "reason": "Toothpicks are typically single-material wood. Per the material rule, single-material wood items map to 'Wood and products of wood and cork (20)'. Plastic variants exist but wood is the default.", "confidence_level": 0.8}
    [16] {"activity": "Rubber and plastic products (25)", "reason": "A nylon pool cleaning brush is primarily plastic (nylon bristles, likely plastic handle). Per the material rule, single-dominant-material plastic items map to 'Rubber and plastic products (25)'.", "confidence_level": 0.65}
    [17] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "Soup spoons are single-material fabricated metal (typically stainless steel). Per the material rule, fabricated metal items map to 'Fabricated metal products (28)'.", "confidence_level": 0.9}
    [18] {"activity": "Furniture; other manufactured goods n.e.c. (36)", "reason": "Fitness equipment is a composite manufactured good (metal, plastic, padding). The activity catalog has no sports/fitness-specific category. Per the composite-goods override, composite manufactured items map to 'Furniture; other manufactured goods n.e.c. (36)'.", "confidence_level": 0.5}
    [19] {"activity": "Chemicals nec", "reason": "SPF15 body oil is a cosmetic chemical product. Per the chemicals rule extended from paint logic, cosmetic chemical products map to 'Chemicals nec'.", "confidence_level": 0.8}
    [20] {"activity": "Electrical machinery and apparatus n.e.c. (31)", "reason": "A ZITON input module is an electrical/electronic component for fire alarm systems. Per the electronics rule, electrical apparatus maps to 'Electrical machinery and apparatus n.e.c. (31)'.", "confidence_level": 0.85}
    [21] {"activity": "Electrical machinery and apparatus n.e.c. (31)", "reason": "An electric insect killer with a fluorescent tube is an electrical appliance. Per the electronics rule, electrical apparatus maps to 'Electrical machinery and apparatus n.e.c. (31)'.", "confidence_level": 0.85}
    [22] {"activity": "Rubber and plastic products (25)", "reason": "Key tags are typically single-material plastic. Per the material rule, single-material plastic items map to 'Rubber and plastic products (25)'.", "confidence_level": 0.75}
    [23] {"activity": "Chemicals nec", "reason": "An invoice from 'Beauty Solution Trading' for cosmetics products. Per the chemicals rule, cosmetic chemical products map to 'Chemicals nec'.", "confidence_level": 0.7}
    [24] {"activity": "Rubber and plastic products (25)", "reason": "A 'Store N Pour' kitchen bottle is single-material plastic. Per the material rule, single-material plastic items map to 'Rubber and plastic products (25)'.", "confidence_level": 0.85}
    [25] {"activity": "Other business services (74)", "reason": "A 'press trip' is a public relations / marketing service for journalists. Per the service rule, PR and marketing services map to 'Other business services (74)'. Travel components are secondary to the PR service nature.", "confidence_level": 0.7}"""

    batch_2_json = """[1] {"activity": "Crops nec", "reason": "A hand bouquet consists of cut/living flowers. Per the special category override, living plants and flowers map to 'Crops nec'.", "confidence_level": 0.9}
    [2] {"activity": "Other business services (74)", "reason": "Emission testing for a generator is a measurement/inspection service. Per the service rule, business support and auditing services map to 'Other business services (74)'.", "confidence_level": 0.6}
    [3] {"activity": "Public administration and defence services; compulsory social security services (75)", "reason": "Work permit government fees are compulsory administrative charges. Per the service rule, visa/government fees map to 'Public administration (75)'.", "confidence_level": 0.95}
    [4] {"activity": "Other land transportation services", "reason": "Cargo repatriation involves moving belongings by road/truck. Per the transport rule, road/truck transport maps to 'Other land transportation services'. Some ambiguity as air freight may be involved.", "confidence_level": 0.55}
    [5] {"activity": "Retail trade services, except of motor vehicles and motorcycles; repair services of personal and household goods (52)", "reason": "Irrigation pump repair is an equipment repair service. Per the service rule, repair services of household and equipment goods map to 'Retail trade services (52)'.", "confidence_level": 0.65}
    [6] {"activity": "Paper and paper products", "reason": "MAXIPULL 2-ply M800 is industrial paper tissue/wiper roll, single-material paper. Per the material rule, single-material paper items map to 'Paper and paper products'.", "confidence_level": 0.8}
    [7] {"activity": "Chemicals nec", "reason": "Shower gel is a cosmetic chemical product (the dispenser packaging is incidental). Per the chemicals rule, cosmetic chemical products map to 'Chemicals nec'.", "confidence_level": 0.8}
    [8] {"activity": "Chemicals nec", "reason": "Jotun Fenomastic is a paint product. Per the special category override, all paint products map to 'Chemicals nec'.", "confidence_level": 0.95}
    [9] {"activity": "Furniture; other manufactured goods n.e.c. (36)", "reason": "Children's toys are composite multi-material manufactured goods (plastic, fabric, electronics). Per the composite-goods override, multi-material manufactured items map to 'Furniture; other manufactured goods n.e.c. (36)'.", "confidence_level": 0.7}
    [10] {"activity": "Retail trade services, except of motor vehicles and motorcycles; repair services of personal and household goods (52)", "reason": "VRV troubleshooting is HVAC equipment repair/diagnosis service. Per the service rule, repair services map to 'Retail trade services (52)'.", "confidence_level": 0.65}
    [11] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "A drill bit is a single-material fabricated metal tool. Per the material rule, fabricated metal items (including hand tool inserts) map to 'Fabricated metal products (28)'.", "confidence_level": 0.85}
    [12] {"activity": "Paraffin Waxes", "reason": "Tea-light candles are made of paraffin wax. Per the petroleum disambiguation rule, paraffin wax and candle wax map to 'Paraffin Waxes'.", "confidence_level": 0.85}
    [13] {"activity": "Textiles (17)", "reason": "Organza bag is a single-material sheer fabric textile product. Per the material rule, textile fabric items (not garments) map to 'Textiles (17)'.", "confidence_level": 0.8}
    [14] {"activity": "Furniture; other manufactured goods n.e.c. (36)", "reason": "An extension pole is a composite manufactured good (aluminum/fiberglass shaft with various fittings). Per the composite-goods override, multi-material manufactured items map to 'Furniture; other manufactured goods n.e.c. (36)'.", "confidence_level": 0.55}
    [15] {"activity": "Textiles (17)", "reason": "A face towel (or mop face towel) is a single-material textile product. Per the material rule, textile fabric items map to 'Textiles (17)'.", "confidence_level": 0.75}
    [16] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "A pipe coupling is a single-material fabricated metal fitting. Per the material rule, fabricated metal items map to 'Fabricated metal products (28)'.", "confidence_level": 0.8}
    [17] {"activity": "Other business services (74)", "reason": "'JSI Rooms AO Q1 2025' in a MARKETING COMMUNICATION category is an under-specified marketing service line item. Per the service rule, marketing/advertising services map to 'Other business services (74)'.", "confidence_level": 0.45}
    [18] {"activity": "Beverages", "reason": "Pepsi is a soft drink. Per the beverage rule, soft drinks/beverages map to 'Beverages'.", "confidence_level": 0.97}
    [19] {"activity": "Leather and leather products (19)", "reason": "A leather tray is a single-material leather product. Per the material rule, single-material leather items map to 'Leather and leather products (19)'.", "confidence_level": 0.85}
    [20] {"activity": "Vegetables, fruit, nuts", "reason": "Cress is a fresh edible plant (microgreen). Per the food rule, fresh minimally-processed produce maps to 'Vegetables, fruit, nuts'.", "confidence_level": 0.85}
    [21] {"activity": "Textiles (17)", "reason": "An eye mask (sleep mask) is a single-material textile product. Per the material rule, textile fabric items map to 'Textiles (17)'. The category 'BANDAGES & DRESSINGS' is misleading; the item is fabric, not medical.", "confidence_level": 0.65}
    [22] {"activity": "Electrical machinery and apparatus n.e.c. (31)", "reason": "An electric insect killer fitting (2x15W) is an electrical apparatus. Per the electronics rule, electrical apparatus maps to 'Electrical machinery and apparatus n.e.c. (31)'.", "confidence_level": 0.85}
    [23] {"activity": "Construction work (45)", "reason": "Concrete work for a gym floor is construction labor. Per the materials-vs-work rule, building/construction work maps to 'Construction work (45)'.", "confidence_level": 0.9}
    [24] {"activity": "Construction work (45)", "reason": "Generator disconnection and shifting is electrical/mechanical installation labor. Per the materials-vs-work rule, installation and building work maps to 'Construction work (45)'. Some ambiguity as it could fit Retail repair services.", "confidence_level": 0.55}
    [25] {"activity": "Meat products nec", "reason": "Haggis is a processed meat product (traditionally sheep offal). Per the food rule, processed meat products not in cattle/pig/poultry sub-categories map to 'Meat products nec'.", "confidence_level": 0.7}"""

    batch_3_json = """[1] {"activity": "Other business services (74)", "reason": "HACCP audit is a food safety certification/auditing service. Per the service rule, auditing and business support services map to 'Other business services (74)'.", "confidence_level": 0.75}
    [2] {"activity": "Office machinery and computers (30)", "reason": "HP Z8 Workstation is a desktop computer. Per the electronics rule, computers and workstations map to 'Office machinery and computers (30)'.", "confidence_level": 0.95}
    [3] {"activity": "Textiles (17)", "reason": "A pencil case is typically a fabric textile item. Per the material rule, fabric items (not garments) map to 'Textiles (17)'. Some ambiguity as it could be composite (fabric + zipper).", "confidence_level": 0.55}
    [4] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "A brass plumbing fitting is a single-material fabricated metal product. Per the material rule, fabricated metal items map to 'Fabricated metal products (28)'. Could alternatively be Copper products since brass is a copper alloy, but the fitting form factor favors the fabricated category.", "confidence_level": 0.7}
    [5] {"activity": "Rubber and plastic products (25)", "reason": "Hard foam (polymer foam) sheets are single-material plastic/rubber-class items. Per the material rule, plastic and rubber items map to 'Rubber and plastic products (25)'.", "confidence_level": 0.85}
    [6] {"activity": "Hotel and restaurant services (55)", "reason": "Venue booking with catering is a hospitality service. Per the service rule, hotel/restaurant/catering services map to 'Hotel and restaurant services (55)'.", "confidence_level": 0.85}
    [7] {"activity": "Public administration and defence services; compulsory social security services (75)", "reason": "A traffic fine is a government-imposed penalty. Per the service rule, compulsory government fees and fines map to 'Public administration (75)'.", "confidence_level": 0.85}
    [8] {"activity": "Other business services (74)", "reason": "Legal/termination consultancy advice is a professional business service. Per the service rule, business advisory services not in finance/audit map to 'Other business services (74)'.", "confidence_level": 0.7}
    [9] {"activity": "Rubber and plastic products (25)", "reason": "A clear squeezy bottle is a single-material plastic item. Per the material rule, plastic items map to 'Rubber and plastic products (25)'.", "confidence_level": 0.85}
    [10] {"activity": "Electrical machinery and apparatus n.e.c. (31)", "reason": "A power bank is a portable battery / electrical apparatus. Per the electronics rule, electrical apparatus maps to 'Electrical machinery and apparatus n.e.c. (31)'.", "confidence_level": 0.75}
    [11] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "GI (Galvanized Iron) return-air duct is single-material fabricated sheet metal. Per the material rule, fabricated metal items map to 'Fabricated metal products (28)'.", "confidence_level": 0.85}
    [12] {"activity": "Food products nec", "reason": "Beyond Meat vegan mince is a packaged plant-based food product not matching specific sub-categories. Per the food rule, packaged food not in a sub-category maps to 'Food products nec'.", "confidence_level": 0.85}
    [13] {"activity": "Food products nec", "reason": "Packaged biscuits are processed food not matching dairy, meat, fish, sugar, rice, oils, or fresh produce. Per the food rule, packaged food not in a specific sub-category maps to 'Food products nec'.", "confidence_level": 0.85}
    [14] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "A scaffolding toe board with fittings is fabricated metal construction equipment. Per the material rule, fabricated metal items map to 'Fabricated metal products (28)'.", "confidence_level": 0.75}
    [15] {"activity": "Other services (93)", "reason": "AV/BGM/ICT for a location is an audio-visual and event services bundle. Per the service rule (and matching gold example for AV+backdrop event services), generic event services map to 'Other services (93)'.", "confidence_level": 0.55}
    [16] {"activity": "Wearing apparel; furs (18)", "reason": "A graffiti cap is headwear/clothing. Per the apparel rule, clothing items map to 'Wearing apparel; furs (18)'.", "confidence_level": 0.9}
    [17] {"activity": "Crops nec", "reason": "A potted Poinsettia is a living plant. Per the special category override, living plants and flowers map to 'Crops nec'.", "confidence_level": 0.95}
    [18] {"activity": "Vegetables, fruit, nuts", "reason": "Fresh basil cress is a minimally-processed edible plant. Per the food rule, fresh produce maps to 'Vegetables, fruit, nuts'.", "confidence_level": 0.85}
    [19] {"activity": "Electrical machinery and apparatus n.e.c. (31)", "reason": "AA batteries are electrical components. Per the electronics rule, electrical apparatus maps to 'Electrical machinery and apparatus n.e.c. (31)'.", "confidence_level": 0.85}
    [20] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "An AC dryer is a fabricated metal component (refrigerant filter-drier). Per the material rule, fabricated metal parts map to 'Fabricated metal products (28)'. Some ambiguity as it could fit motor vehicles if automotive.", "confidence_level": 0.5}
    [21] {"activity": "Other waste for treatment: waste water treatment", "reason": "Recycling of cardboard waste is a waste management/treatment service. The activity catalog's three waste activities are landfill (food, hazardous) and general 'Other waste for treatment'; recycling falls under the latter as the closest match.", "confidence_level": 0.45}
    [22] {"activity": "Vegetables, fruit, nuts", "reason": "Frozen broad beans are minimally-processed vegetables. Per the food rule, fresh/frozen produce maps to 'Vegetables, fruit, nuts'.", "confidence_level": 0.85}
    [23] {"activity": "Furniture; other manufactured goods n.e.c. (36)", "reason": "A custom baby chair cushion in PU leather with velcro + buckle + strap is a composite multi-material manufactured good. Per the composite-goods override, multi-material items map to 'Furniture; other manufactured goods n.e.c. (36)'.", "confidence_level": 0.7}
    [24] {"activity": "Food products nec", "reason": "Cupcakes are packaged baked food not in a specific sub-category. Per the food rule, packaged food not in dairy/meat/fish/produce/sugar/rice/oils maps to 'Food products nec'.", "confidence_level": 0.8}
    [25] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "A stainless steel reducing nipple is a single-material fabricated metal pipe fitting. Per the material rule, fabricated metal items map to 'Fabricated metal products (28)'.", "confidence_level": 0.9}"""

    batch_4_json = """[1] {"activity": "Chemicals nec", "reason": "Electric mosquito repellent refill is a chemical pesticide product. Per the chemicals rule, pesticides and chemical repellents map to 'Chemicals nec'.", "confidence_level": 0.85}
    [2] {"activity": "Furniture; other manufactured goods n.e.c. (36)", "reason": "A NORDAQ medium water filter is a composite manufactured good (plastic housing + filter media). Per the composite-goods override, multi-material manufactured items map to 'Furniture; other manufactured goods n.e.c. (36)'.", "confidence_level": 0.55}
    [3] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "A server rack mounting kit is single-material fabricated metal hardware. Per the material rule, fabricated metal items map to 'Fabricated metal products (28)'.", "confidence_level": 0.6}
    [4] {"activity": "Other business services (74)", "reason": "Recruitment agency fees are business services for hiring. Per the service rule, business advisory and HR services map to 'Other business services (74)'.", "confidence_level": 0.85}
    [5] {"activity": "Retail trade services, except of motor vehicles and motorcycles; repair services of personal and household goods (52)", "reason": "Overhead crane rectification is equipment repair service. Per the service rule, equipment repair maps to 'Retail trade services (52)'.", "confidence_level": 0.6}
    [6] {"activity": "Other business services (74)", "reason": "Printer contract overage charge is a business service line item for managed print services. Per the service rule, business support services map to 'Other business services (74)'.", "confidence_level": 0.6}
    [7] {"activity": "Paper and paper products", "reason": "Printed envelopes are paper office supplies. Per the material rule, paper items map to 'Paper and paper products'.", "confidence_level": 0.6}
    [8] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "A hand tube cutter (non-powered) is single-material fabricated metal. Per the material rule, fabricated metal hand tools map to 'Fabricated metal products (28)'.", "confidence_level": 0.75}
    [9] {"activity": "Chemicals nec", "reason": "A gentle exfoliator is a cosmetic chemical product. Per the chemicals rule, cosmetic chemical products map to 'Chemicals nec'.", "confidence_level": 0.85}
    [10] {"activity": "Insurance and pension funding services, except compulsory social security services (66)", "reason": "Medical/life insurance member addition is an insurance service. Per the service rule, insurance services map to 'Insurance and pension funding services (66)'.", "confidence_level": 0.85}
    [11] {"activity": "Food products nec", "reason": "Packaged saffron is a processed/dried food spice product, not fresh produce. Per the food rule, packaged food not in a specific sub-category maps to 'Food products nec'.", "confidence_level": 0.55}
    [12] {"activity": "Chemicals nec", "reason": "Printer ink is a chemical product (pigments/dyes in solvent). Per the chemicals rule, chemical products like ink map to 'Chemicals nec'.", "confidence_level": 0.8}
    [13] {"activity": "Paper and paper products", "reason": "Kraft mini tubs are primarily paper (kraft paperboard) with secondary PET plastic lids. Per the material rule, the dominant material is paper.", "confidence_level": 0.55}
    [14] {"activity": "Textiles (17)", "reason": "A 100% cotton towel is single-material textile fabric. Per the material rule, textile fabric items map to 'Textiles (17)'.", "confidence_level": 0.9}
    [15] {"activity": "Other non-metallic mineral products", "reason": "A sanding disc with P80 grit abrasive is an abrasive non-metallic mineral product. Per the materials rule, non-metallic mineral products map to 'Other non-metallic mineral products'.", "confidence_level": 0.55}
    [16] {"activity": "Fabricated metal products, except machinery and equipment (28)", "reason": "A stainless steel handle for a chiller unit is single-material fabricated metal. Per the material rule, fabricated metal hardware maps to 'Fabricated metal products (28)'.", "confidence_level": 0.9}
    [17] {"activity": "Furniture; other manufactured goods n.e.c. (36)", "reason": "An AHU (Air Handling Unit) filter is a composite manufactured good (synthetic media + frame). Per the composite-goods override, multi-material manufactured items map to 'Furniture; other manufactured goods n.e.c. (36)'.", "confidence_level": 0.55}
    [18] {"activity": "Food products nec", "reason": "Guacamole is a processed/prepared food (mashed, mixed). Per the food rule, processed packaged food not in a specific sub-category maps to 'Food products nec'.", "confidence_level": 0.7}
    [19] {"activity": "Computer and related services (72)", "reason": "Oaky is an online hotel upselling SaaS platform. Per the service rule, IT/software/SaaS services map to 'Computer and related services (72)'.", "confidence_level": 0.7}
    [20] {"activity": "Crops nec", "reason": "Cut red roses are living/cut flowers. Per the special category override, living plants and flowers map to 'Crops nec'.", "confidence_level": 0.95}
    [21] {"activity": "Construction work (45)", "reason": "FCU (Fan Coil Unit) disconnection, relocation, and modification is HVAC installation/building work. Per the materials-vs-work rule, installation and building work maps to 'Construction work (45)'.", "confidence_level": 0.8}
    [22] {"activity": "Rubber and plastic products (25)", "reason": "An HDPE (High-Density Polyethylene) pipe roll is single-material plastic. Per the material rule, plastic items map to 'Rubber and plastic products (25)'.", "confidence_level": 0.9}
    [23] {"activity": "Wood and products of wood and cork (except furniture); articles of straw and plaiting materials (20)", "reason": "A billiards cue stick is primarily single-material wood (with a small leather tip). Per the material rule, dominant-wood items map to 'Wood and products of wood and cork (20)'.", "confidence_level": 0.65}
    [24] {"activity": "Wearing apparel; furs (18)", "reason": "A uniform shirt is a garment. Per the apparel rule, clothing and uniforms map to 'Wearing apparel; furs (18)'.", "confidence_level": 0.95}
    [25] {"activity": "Beverages", "reason": "Pisco is a Peruvian alcoholic spirit (beverage). Per the beverage rule, alcoholic drinks map to 'Beverages'.", "confidence_level": 0.97}"""

    claude_audit_records = []
    for blob, offset in [(batch_1_json,0),(batch_2_json,25),(batch_3_json,50),(batch_4_json,75)]:
        for line in blob.strip().split("\n"):
            m = re.match(r'\[(\d+)\]\s*(\{.*\})', line)
            if not m: continue
            idx = int(m.group(1)) - 1 + offset
            parsed = json.loads(m.group(2))
            row = audit_sample.iloc[idx]
            claude_audit_records.append({'item_id': row['item_id'],
                'item_description': row['item_description'],
                'claude_activity':   parsed['activity'],
                'claude_reason':     parsed['reason'],
                'claude_confidence': parsed['confidence_level']})
    claude_audit_df = pd.DataFrame(claude_audit_records)
    claude_audit_df.to_csv(f'{ROOT}/silver_labels/claude_audit.csv', index=False)
    print('Rebuilt Claude audit:', len(claude_audit_df))

In [17]:
# Claude <-> Qwen agreement on the 100-item audit (label-quality signal)
audit = claude_audit_df.merge(qwen_df[['item_id','qwen_activity','qwen_confidence']], on='item_id')
audit['agree'] = audit['claude_activity'] == audit['qwen_activity']
print(f'Claude <-> Qwen agreement: {audit["agree"].mean():.1%}  ({audit["agree"].sum()}/{len(audit)})')
audit.to_csv(f'{ROOT}/silver_labels/audit_comparison.csv', index=False)

Claude <-> Qwen agreement: 62.0%  (62/100)


**Gemini cross-check.** The other 500 items were labeled with **Gemini 2.5 Pro** via AI Studio (web), pasted back in five 100-item batches and parsed to `gemini_labels_final.csv`. The full responses are archived in `silver_labels/`; we load the parsed result here.

In [18]:
gemini_df = pd.read_csv(f'{ROOT}/silver_labels/gemini_labels_final.csv')
print('Gemini cross-check labels:', len(gemini_df))
gemini_df[['item_description','gemini_activity','gemini_confidence']].head(3)

Gemini cross-check labels: 500


,item_description,gemini_activity,gemini_confidence
0,Tax Invoice 138 - 26385 - Member Addition,"Insurance and pension funding services, except...",0.95
1,Texture Xanthan N° 10 PCB PCB05601 1kg France,Food products nec,0.95
2,Travel Media - LTM participation Moscow Septem...,Other business services (74),0.85


## 4 · Dataset assembly

Merge the two teachers on each item. Where they **agree**, keep the Qwen label (consistent). Where they **disagree** (≈51% of the 500), each case was hand-adjudicated — `verdicts` records whether Qwen (`Q`) or Gemini (`G`) won. The 100 Claude-audited items keep their Qwen labels. Final 600 → stratified `train / val / test_silver`.

In [19]:
qwen_for_merge = qwen_df[['item_description', 'qwen_activity', 'qwen_confidence']]
merged = gemini_df.merge(qwen_for_merge, on='item_description', how='left')
merged['agree'] = merged['gemini_activity'] == merged['qwen_activity']
agreement_rate = merged['agree'].mean()
print(f"Qwen vs Gemini agreement: {agreement_rate:.1%}  ({merged['agree'].sum()} / {len(merged)})")
print(f"Unmatched: {merged['qwen_activity'].isna().sum()}")

Qwen vs Gemini agreement: 49.2%  (246 / 500)
Unmatched: 0


In [20]:
# Verdicts: key = disagree_df row index (0-based), value = 'Q' or 'G'
verdicts = {
    0:'G', 1:'Q', 2:'G', 3:'G', 4:'G', 5:'G', 6:'G', 7:'G', 8:'G', 9:'G',
   10:'G', 11:'G', 12:'G', 13:'G', 14:'G', 15:'Q', 16:'G', 17:'G', 18:'G', 19:'G',
   20:'G', 21:'G', 22:'Q', 23:'G', 24:'Q', 25:'Q', 26:'G', 27:'G', 28:'G', 29:'G',
   30:'G', 31:'Q', 32:'G', 33:'Q', 34:'G', 35:'G', 36:'G', 37:'G', 38:'Q', 39:'Q',
   40:'G', 41:'Q', 42:'G', 43:'G', 44:'G', 45:'G', 46:'Q', 47:'G', 48:'Q', 49:'G',
   50:'G', 51:'G', 52:'G', 53:'Q', 54:'G', 55:'G', 56:'Q', 57:'G', 58:'G', 59:'G',
   60:'G', 61:'G', 62:'G', 63:'G', 64:'Q', 65:'G', 66:'G', 67:'Q', 68:'G', 69:'Q',
   70:'G', 71:'G', 72:'G', 73:'Q', 74:'G', 75:'Q', 76:'Q', 77:'Q', 78:'Q', 79:'Q',
   80:'Q', 81:'Q', 82:'Q', 83:'Q', 84:'Q', 85:'Q', 86:'Q', 87:'Q', 88:'Q', 89:'Q',
   90:'Q', 91:'Q', 92:'Q', 93:'Q', 94:'Q', 95:'Q', 96:'Q', 97:'Q', 98:'Q', 99:'Q',
  100:'Q', 101:'Q', 102:'Q', 103:'Q', 104:'Q', 105:'Q', 106:'Q', 107:'Q', 108:'Q', 109:'Q',
  110:'Q', 111:'Q', 112:'Q', 113:'Q', 114:'Q', 115:'Q', 116:'Q', 117:'Q', 118:'Q', 119:'Q',
  120:'Q', 121:'Q', 122:'Q', 123:'Q', 124:'Q', 125:'Q', 126:'Q', 127:'Q', 128:'Q', 129:'Q',
  130:'Q', 131:'Q', 132:'Q', 133:'Q', 134:'Q', 135:'Q', 136:'Q', 137:'Q', 138:'Q', 139:'Q',
  140:'Q', 141:'Q', 142:'Q', 143:'Q', 144:'Q', 145:'Q', 146:'Q', 147:'Q', 148:'Q', 149:'Q',
  150:'Q', 151:'Q', 152:'Q', 153:'Q', 154:'Q', 155:'Q', 156:'Q', 157:'Q', 158:'Q', 159:'Q',
  160:'Q', 161:'Q', 162:'Q', 163:'Q', 164:'Q', 165:'Q', 166:'Q', 167:'Q', 168:'Q', 169:'Q',
  170:'Q', 171:'G', 172:'G', 173:'Q', 174:'G', 175:'G', 176:'G', 177:'Q', 178:'G', 179:'G',
  180:'G', 181:'G', 182:'Q', 183:'Q', 184:'G', 185:'Q', 186:'Q', 187:'G', 188:'G', 189:'G',
  190:'G', 191:'G', 192:'G', 193:'Q', 194:'G', 195:'G', 196:'G', 197:'G', 198:'Q', 199:'Q',
  200:'G', 201:'G', 202:'Q', 203:'G', 204:'G', 205:'G', 206:'G', 207:'G', 208:'G', 209:'G',
  210:'Q', 211:'Q', 212:'Q', 213:'G', 214:'G', 215:'G', 216:'G', 217:'G', 218:'G', 219:'G',
  220:'G', 221:'G', 222:'G', 223:'G', 224:'G', 225:'G', 226:'G', 227:'G', 228:'G', 229:'G',
  230:'G', 231:'G', 232:'G', 233:'G', 234:'G', 235:'G', 236:'G', 237:'G', 238:'G', 239:'G',
  240:'G', 241:'Q', 242:'G', 243:'G', 244:'G', 245:'G', 246:'G', 247:'Q', 248:'G', 249:'Q',
  250:'G', 251:'G', 252:'G', 253:'G',
}

# Apply verdicts to disagreements
def pick_label(row):
    idx = row.name
    v = verdicts.get(idx, 'Q')  # default Q if missing
    if v == 'G':
        return row['gemini_activity'], row['gemini_confidence'], 'gemini_audited'
    else:
        return row['qwen_activity'], row['qwen_confidence'], 'qwen_audited'

disagree_df_indexed = merged[~merged['agree']].reset_index(drop=True).copy()
results = disagree_df_indexed.apply(pick_label, axis=1, result_type='expand')
results.columns = ['final_activity', 'final_confidence', 'label_source']
disagree_df_indexed[['final_activity','final_confidence','label_source']] = results

g_count = sum(1 for v in verdicts.values() if v == 'G')
q_count = sum(1 for v in verdicts.values() if v == 'Q')
print(f"Verdicts applied: {g_count} Gemini wins, {q_count} Qwen wins")

Verdicts applied: 124 Gemini wins, 130 Qwen wins


In [21]:
# Combine agreed rows + adjudicated disagreements
agree_rows = merged[merged['agree']].copy()
agree_rows['final_activity']   = agree_rows['qwen_activity']
agree_rows['final_confidence'] = agree_rows['qwen_confidence']
agree_rows['label_source']     = 'qwen_gemini_agree'

cols = ['item_description','main_category','category_3','final_activity','final_confidence','label_source']
silver_500 = pd.concat([agree_rows[cols], disagree_df_indexed[cols]], ignore_index=True)

# Add the 100 Claude-audited items (Qwen labels; Claude was auditor, not labeler)
claude_ids = set(claude_audit_df['item_id'])
claude_100 = (qwen_df[qwen_df['item_id'].isin(claude_ids)]
              [['item_description','main_category','category_3','qwen_activity','qwen_confidence']]
              .rename(columns={'qwen_activity':'final_activity','qwen_confidence':'final_confidence'}))
claude_100['label_source'] = 'qwen_claude_audited'

silver_all = pd.concat([silver_500, claude_100], ignore_index=True)
print('Total silver labels:', len(silver_all))
print(silver_all['label_source'].value_counts().to_string())
print(f'Activity coverage: {silver_all["final_activity"].nunique()}/83')

Total silver labels: 600
label_source
qwen_gemini_agree      246
qwen_audited           130
gemini_audited         124
qwen_claude_audited    100
Activity coverage: 56/83


In [22]:
from sklearn.model_selection import train_test_split

# Freeze the split: load it if it already exists so train/test stay identical across runs
# (prevents the adapter being evaluated on items it was trained on). Regenerate only if missing.
split_dir = f"{DRIVE_ROOT}/data/splits"
_have = all(os.path.exists(f"{split_dir}/{n}.csv") for n in ["train","val","test_silver"])
if _have and not REGENERATE_SILVER:
    train       = pd.read_csv(f"{split_dir}/train.csv")
    val         = pd.read_csv(f"{split_dir}/val.csv")
    test_silver = pd.read_csv(f"{split_dir}/test_silver.csv")
    print(f"Loaded frozen splits  Train {len(train)} | Val {len(val)} | Test {len(test_silver)}")
else:
    counts = silver_all['final_activity'].value_counts()
    rare   = counts[counts < 4].index
    common    = silver_all[~silver_all['final_activity'].isin(rare)]
    rare_rows = silver_all[silver_all['final_activity'].isin(rare)]
    train_c, temp = train_test_split(common, test_size=0.2, stratify=common['final_activity'], random_state=42)
    _tc = temp['final_activity'].value_counts(); _still = _tc[_tc < 2].index
    extra_to_train = temp[temp['final_activity'].isin(_still)]
    temp_safe      = temp[~temp['final_activity'].isin(_still)]
    val, test_silver = train_test_split(temp_safe, test_size=0.5, stratify=temp_safe['final_activity'], random_state=42)
    train = pd.concat([train_c, rare_rows, extra_to_train], ignore_index=True)
    os.makedirs(split_dir, exist_ok=True)
    train.to_csv(f"{split_dir}/train.csv", index=False)
    val.to_csv(f"{split_dir}/val.csv", index=False)
    test_silver.to_csv(f"{split_dir}/test_silver.csv", index=False)
    print(f"Created + froze splits  Train {len(train)} | Val {len(val)} | Test {len(test_silver)}")

Loaded frozen splits  Train 498 | Val 51 | Test 51


## 5 · Baselines

Three reference points, all evaluated on the same gold-20 and silver-52 test sets:
1. **Embedding-NN** (BAAI/bge-large) — non-LLM lookup
2. **Zero-shot** — Qwen-14B with the activity list only (no rule book)
3. **Base + rule book** — Qwen-14B with the full prompt, no fine-tuning

In [23]:
# Load the student base model once (8192 ctx — the rule-book prompt is ~4.6k tokens)
model_base, tok_base = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-14B-Instruct-bnb-4bit", max_seq_length=8192, load_in_4bit=True)
FastLanguageModel.for_inference(model_base)
print('Qwen-14B base loaded.')

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/210k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-14B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Qwen-14B base loaded.


In [24]:
def run_inference(model, tok, df, msg_fn, gold_col):
    """Greedy-decode each row, parse JSON, return (preds_df, exact_acc, parse_rate)."""
    STOP = [tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")]
    recs = []
    for r in df.itertuples():
        msgs = msg_fn(r.item_description, getattr(r,'main_category',None), getattr(r,'category_3',None))
        inp  = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                       return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(input_ids=inp, max_new_tokens=200, do_sample=False,
                                  eos_token_id=STOP, pad_token_id=tok.eos_token_id)
        gen = tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
        p = parse_json_output(gen)
        recs.append({'item_description': r.item_description,
                     'gold_activity':   getattr(r, gold_col),
                     'pred_activity':   p.get('activity')         if p else None,
                     'pred_confidence': p.get('confidence_level') if p else None,
                     'parse_ok': p is not None})
    out_df = pd.DataFrame(recs)
    return out_df, (out_df.gold_activity == out_df.pred_activity).mean(), out_df.parse_ok.mean()

def sector_acc(df):
    return np.mean([sector_map.get(p) == sector_map.get(g)
                    for p, g in zip(df.pred_activity, df.gold_activity)])

In [25]:
# Zero-shot and base+rule-book on both test sets
zs_gold_df,  zs_gold,  _ = run_inference(model_base, tok_base, gold_test,    build_messages_zeroshot, 'activity')
zs_silv_df,  zs_silv,  _ = run_inference(model_base, tok_base, test_silver,  build_messages_zeroshot, 'final_activity')
base_gold_df, base_gold, base_gold_parse = run_inference(model_base, tok_base, gold_test,   build_messages, 'activity')
base_silv_df, base_silv, _              = run_inference(model_base, tok_base, test_silver, build_messages, 'final_activity')

print(f'Zero-shot        — gold {zs_gold:.1%} | silver {zs_silv:.1%}')
print(f'Base+rule book   — gold {base_gold:.1%} (rule book partly derived from gold → optimistic) | silver {base_silv:.1%}')
for d,n in [(base_gold_df,'baseline_predictions'),(zs_gold_df,'zeroshot_predictions'),(base_silv_df,'baseline_silver_predictions')]:
    d.to_csv(f'{ROOT}/artifacts/{n}.csv', index=False)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Zero-shot        — gold 55.0% | silver 45.1%
Base+rule book   — gold 95.0% (rule book partly derived from gold → optimistic) | silver 58.8%


## 6 · Fine-tune (LoRA)

LoRA via Unsloth + TRL `SFTTrainer`: rank 16 / α 16 on all attention + MLP projections, 2 epochs, lr 2e-4 cosine, batch 2 × grad-accum 4, packing on. Training prompts carry the **same** system prompt used at inference, so the model learns to apply the rule book to real items. *(Loads the saved adapter unless `RETRAIN=True`.)*

In [26]:
# Free the base model before loading the trainable copy
for v in ['model_base','tok_base','teacher_qwen','tok_qwen']:
    if v in dir(): exec(f'del {v}')
gc.collect(); torch.cuda.empty_cache()
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

GPU free: 41.8 GB


In [27]:
lora_path = f'{ROOT}/lora_adapter'

if os.path.exists(lora_path) and not RETRAIN:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=lora_path, max_seq_length=8192, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    print('Loaded saved LoRA adapter from', lora_path)
else:
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen2.5-14B-Instruct-bnb-4bit", max_seq_length=8192, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=42)

    def format_example(row):
        msgs = build_messages(row['item_description'], row['main_category'], row['category_3'])
        msgs.append({"role":"assistant",
            "content": f'{{"activity": "{row["final_activity"]}", "reason": "Mapped per rule book.", "confidence_level": {row["final_confidence"]}}}'})
        return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

    train_ds = Dataset.from_list([format_example(r) for _, r in train.iterrows()])
    val_ds   = Dataset.from_list([format_example(r) for _, r in val.iterrows()])
    print(f'Train {len(train_ds)} | Val {len(val_ds)}')

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
        args=SFTConfig(
            output_dir=f'{ROOT}/checkpoints',
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
            fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10, eval_strategy="steps", eval_steps=50,
            save_strategy="steps", save_steps=50,
            load_best_model_at_end=True, metric_for_best_model="eval_loss",
            dataset_text_field="text", max_seq_length=8192, packing=True, report_to="none"))
    trainer.train()
    model.save_pretrained(lora_path); tokenizer.save_pretrained(lora_path)
    FastLanguageModel.for_inference(model)
    print('Trained + saved LoRA adapter to', lora_path)

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/Qwen2.5-14B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.1 patched 48 layers with 48 QKV layers, 48 O layers and 48 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train 498 | Val 51


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 498 | Num Epochs = 2 | Total steps = 126
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 68,812,800 of 14,838,846,464 (0.46% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.019849,0.019557
100,0.016845,0.018359
126,0.016264,0.018299


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/carbonsifr/checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/carbonsifr/checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/carbonsifr/checkpoints/checkpoint-126/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/carbonsifr/lora_adapter/tokenizer_config.json.


Trained + saved LoRA adapter to /content/drive/MyDrive/carbonsifr/lora_adapter


## 7 · Evaluation & comparison

In [28]:
ft_gold_df,  ft_gold,  ft_gold_parse = run_inference(model, tokenizer, gold_test,   build_messages, 'activity')
ft_silv_df,  ft_silv,  ft_silv_parse = run_inference(model, tokenizer, test_silver, build_messages, 'final_activity')
print(f'Fine-tuned — gold {ft_gold:.1%} | silver {ft_silv:.1%}  (base+rulebook silver was {base_silv:.1%})')
ft_gold_df.to_csv(f'{ROOT}/artifacts/finetuned_predictions.csv', index=False)
ft_silv_df.to_csv(f'{ROOT}/artifacts/finetuned_silver_predictions.csv', index=False)

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

Fine-tuned — gold 95.0% | silver 76.5%  (base+rulebook silver was 58.8%)


In [29]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Encode training set
train_descriptions = train['item_description'].tolist()
train_activities = train['final_activity'].tolist()
train_embeddings = embedder.encode(train_descriptions, normalize_embeddings=True, show_progress_bar=True)

# Eval on gold-20
gold_descriptions = gold_test['item_description'].tolist()
gold_embeddings = embedder.encode(gold_descriptions, normalize_embeddings=True)

scores = gold_embeddings @ train_embeddings.T  # cosine similarity
nn_preds_gold = [train_activities[i] for i in scores.argmax(axis=1)]
nn_exact_gold = np.mean([p == g for p, g in zip(nn_preds_gold, gold_test['activity'].tolist())])
print(f"Embedding NN (gold-20): {nn_exact_gold:.1%}")

# Eval on silver-52
silver_descriptions = test_silver['item_description'].tolist()
silver_embeddings = embedder.encode(silver_descriptions, normalize_embeddings=True)
scores_s = silver_embeddings @ train_embeddings.T
nn_preds_silver = [train_activities[i] for i in scores_s.argmax(axis=1)]
nn_exact_silver = np.mean([p == g for p, g in zip(nn_preds_silver, test_silver['final_activity'].tolist())])
print(f"Embedding NN (silver-52): {nn_exact_silver:.1%}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding NN (gold-20): 30.0%
Embedding NN (silver-52): 49.0%


In [30]:
# Sector-level accuracy for the NN baseline
nn_sector_gold   = np.mean([sector_map.get(p)==sector_map.get(g) for p,g in zip(nn_preds_gold,   gold_test['activity'])])
nn_sector_silver = np.mean([sector_map.get(p)==sector_map.get(g) for p,g in zip(nn_preds_silver, test_silver['final_activity'])])

comparison = pd.DataFrame([
    {'system':'Embedding NN',          'gold_exact':nn_exact_gold,'gold_sector':nn_sector_gold,
                                       'silver_exact':nn_exact_silver,'silver_sector':nn_sector_silver},
    {'system':'Qwen-14B zero-shot',    'gold_exact':zs_gold,'gold_sector':sector_acc(zs_gold_df),
                                       'silver_exact':zs_silv,'silver_sector':sector_acc(zs_silv_df)},
    {'system':'Qwen-14B base+rulebook','gold_exact':base_gold,'gold_sector':sector_acc(base_gold_df),
                                       'silver_exact':base_silv,'silver_sector':sector_acc(base_silv_df)},
    {'system':'Qwen-14B + LoRA',       'gold_exact':ft_gold,'gold_sector':sector_acc(ft_gold_df),
                                       'silver_exact':ft_silv,'silver_sector':sector_acc(ft_silv_df)},
])
for c in comparison.columns[1:]:
    comparison[c] = (comparison[c]*100).round(1)
comparison.to_csv(f'{ROOT}/artifacts/comparison_table.csv', index=False)
print('Note: gold-exact for base+rulebook is optimistic (rule book derived from the gold items).')
comparison

Note: gold-exact for base+rulebook is optimistic (rule book derived from the gold items).


,system,gold_exact,gold_sector,silver_exact,silver_sector
0,Embedding NN,30.0,40.0,49.0,56.9
1,Qwen-14B zero-shot,55.0,65.0,45.1,58.8
2,Qwen-14B base+rulebook,95.0,100.0,58.8,68.6
3,Qwen-14B + LoRA,95.0,95.0,76.5,84.3


## 8 · Demo — classify a single item

In [31]:
def classify(description, main_category="unspecified", category_3="unspecified"):
    msgs = build_messages(description, main_category, category_3)
    inp  = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                         return_tensors="pt").to("cuda")
    STOP = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]
    with torch.no_grad():
        out = model.generate(input_ids=inp, max_new_tokens=200, do_sample=False,
                             eos_token_id=STOP, pad_token_id=tokenizer.eos_token_id)
    return parse_json_output(tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True))

for ex in ["Fresh mango 1kg Thailand",
           "Office repainting service - 3 walls",
           "DLINK Cat 6 cable 5 Mtr"]:
    print(ex, "->", classify(ex))

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

Fresh mango 1kg Thailand -> {'activity': 'Vegetables, fruit, nuts', 'reason': 'Mapped per rule book.', 'confidence_level': 0.95}


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Office repainting service - 3 walls -> {'activity': 'Construction work (45)', 'reason': 'Mapped per rule book.', 'confidence_level': 0.85}
DLINK Cat 6 cable 5 Mtr -> {'activity': 'Electrical machinery and apparatus n.e.c. (31)', 'reason': 'Mapped per rule book.', 'confidence_level': 0.95}
